In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q3-stage3-2026")

print("Path to dataset files:", path) #d f n w

In [ ]:
def remap_mask(mask):
    # Remaps a mask's pixel values to a consecutive range starting at 0
    mask = mask.long()
    unique_values = torch.unique(mask)
    remapped_mask = torch.zeros_like(mask)

    for new_val, old_val in enumerate(sorted(unique_values.tolist())):
        remapped_mask[mask == old_val] = new_val

    return remapped_mask

In [ ]:
# TO DO
from torch.utils.data import Dataset
from PIL import Image
import glob
import os
import torchvision.transforms as transforms

class CustomDataset(Dataset): #root_dir = path
    def __init__(self, image_dir, mask_dir, transform=None, target_transform=None):
        self.image_paths = glob.glob(os.path.join(image_dir, "*.jpg"))  # Get all image paths
        self.mask_paths = glob.glob(os.path.join(mask_dir, "*.png"))  # Get all mask paths
        self.transform = transform  # Transformations
        self.class_labels = {
            "Background (waterbody)": 0, "Human divers": 1, "Aquatic plants and sea-grass": 2,
            "Wrecks and ruins": 3, "Robots (AUVs/ROVs/instruments)": 4, "Reefs and invertebrates": 5,
            "Fish and vertebrates": 6, "Sea-floor and rocks": 7
        }

        self.image_paths = []
        self.labels = []
        #for class_name, label in self.class_labels.items(): #understand how this works #
         #   for class_path in os.listdir(path):
             # for image_path in os.listdir(os.path.join(class_path)):
               #   image_path_list.append(os.path.join(self.root_dir,class_path, image_path))




        self.image_paths.sort()
        self.mask_paths.sort()

        self.transform = transform
        self.target_transform = target_transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        # 🔹 Load the image and mask
        image = Image.open(self.image_paths[idx]).convert("RGB")
        mask = Image.open(self.mask_paths[idx]).convert("L")

        # 🔹 Apply transformations for image
        if self.transform:
            image = self.transform(image)

        # 🔹 Apply transformations for mask
        if self.target_transform:
            mask = self.target_transform(mask)

        mask = remap_mask(mask)

        return image, mask

from torch.utils.data import DataLoader
from torch import nn

# Define transforms for images and masks
image_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((256, 256)),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # Standard ImageNet normalization
])

mask_transforms = transforms.Compose([
    transforms.Resize((256, 256), interpolation=transforms.InterpolationMode.NEAREST),  # Keep segmentation masks intact
    transforms.PILToTensor(),
])


train_image_dir = os.path.join(path, "train", "images")
train_mask_dir = os.path.join(path, "train", "masks")

test_image_dir = os.path.join(path, "val", "images")
test_mask_dir = os.path.join(path, "val", "masks")

train_dataset = CustomDataset(train_image_dir, train_mask_dir, transform=image_transforms, target_transform=mask_transforms)
test_dataset = CustomDataset(test_image_dir, test_mask_dir, transform=image_transforms, target_transform=mask_transforms)


# Create Train & Test DataLoaders
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False, num_workers=2)

# Check dataset sizes
print(f"Training Samples: {len(train_dataset)}, Testing Samples: {len(test_dataset)}")


# Create DataLoader
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True, num_workers=2)

import matplotlib.pyplot as plt

# Display some images with their masks
for i in range(10,13):
    img, mask = train_dataset[i]
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(img.permute(1, 2, 0))  # Convert (C, H, W) to (H, W, C)
    axes[0].set_title("Image")
    axes[0].axis("off")
    axes[1].imshow(mask.squeeze(), cmap="gray")
    axes[1].set_title("Segmentation Mask")
    axes[1].axis("off")
    plt.show()


In [ ]:
# TO DO

import segmentation_models_pytorch as smp #sorry couldn't find how to donwload it

# Define U-Net Model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = smp.Unet(
    encoder_name="efficientnet-b1",  # Pretrained encoder (backbone)
    encoder_weights="imagenet",  # Use ImageNet weights
    in_channels=3,  # RGB images
    classes=5,
).to(device)


In [ ]:
# TO DO
import torch.optim as optim
import torch.nn.functional as F
from tqdm import tqdm

# 🔹 Training Loop
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0

    for images, masks in tqdm(dataloader):
        images, masks = images.to(device), masks.to(device).squeeze(dim=1).to(torch.long)

        outputs = model(images)
        loss = criterion(outputs, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

# 🔹 Validation Loop
def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for images, masks in dataloader:
            images, masks = images.to(device), masks.to(device).squeeze(dim=1).to(torch.long)

            outputs = model(images)
            loss = criterion(outputs, masks)
            total_loss += loss.item()

    return total_loss / len(dataloader)

In [ ]:
# TO DO
import torch
from torch import nn
# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.0001)

num_epochs = 10  # Define number of epochs
train_losses = []
val_losses = []

# Training Loop
for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss = validate(model, test_loader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch {epoch+1}/{num_epochs}: Train Loss = {train_loss:.4f}, Val Loss = {val_loss:.4f}")


In [ ]:
# TO DO
import matplotlib.pyplot as plt

plt.plot(range(1, num_epochs+1), train_losses, label="Train Loss", marker='o')
plt.plot(range(1, num_epochs+1), val_losses, label="Validation Loss", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.show()